# Gridlock 2.0 — Detection Fine-Tune (Kaggle, T4×2 GPU)

Fine-tunes two detectors on **IDD (India Driving Dataset)** for the 15 Indian-traffic classes
(incl. **auto-rickshaw / vehicle-fallback** that zero-shot COCO cannot see):

1. **YOLOv11-l** — fast comparator (Ultralytics, AGPL → internal benchmark only, not shipped).
2. **RF-DETR-Large** — the shippable detector (Apache-2.0; master-design §3.2 / §8 primary).

### Why GPU, not TPU (v5e-8)
Both models are CUDA-PyTorch with custom ops and **cannot run on TPU**:
- **Ultralytics YOLO** has no TPU/`torch_xla` path (CUDA/MPS/CPU only).
- **RF-DETR** uses deformable-attention **custom CUDA kernels** with no XLA implementation.

→ Use the **T4×2** accelerator (Settings → Accelerator → GPU T4 x2). 2×16 GB is ample.

**Before running:** Settings → Accelerator = **GPU T4 x2**, and add your uploaded IDD dataset
(it appears under `/kaggle/input/<your-dataset-name>/`). Set `IDD_ROOT` in the Config cell.

## 0. Install dependencies

In [ ]:
# RF-DETR (+train extras) and Ultralytics. Torch/CUDA are preinstalled on Kaggle GPU images.
!pip -q install "rfdetr[train]" ultralytics pycocotools 2>/dev/null
print('installed')

In [ ]:
import torch, os, json, glob, shutil, time
import xml.etree.ElementTree as ET
from pathlib import Path
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '| GPUs', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  cuda:{i}  {p.name}  {round(p.total_memory/1024**3,1)} GB')

## 1. Config — EDIT `IDD_ROOT`
Point this at the folder that contains `JPEGImages/`, `Annotations/`, and `train.txt`/`val.txt`.

In [ ]:
# >>> EDIT THIS to your uploaded dataset path (look under /kaggle/input/...) <<<
IDD_ROOT = '/kaggle/input/idd-detection/IDD_Detection'

# Quick smoke test on a subset first? set e.g. 3000; None = full dataset (best result).
SUBSET = None

WORK      = Path('/kaggle/working')
COCO_DIR  = WORK / 'idd_coco'
YOLO_DIR  = WORK / 'idd_yolo'
assert Path(IDD_ROOT).exists(), f'IDD_ROOT not found: {IDD_ROOT}  (check /kaggle/input)'
for sp in ('train','val'):
    print(sp+'.txt:', (Path(IDD_ROOT)/f'{sp}.txt').exists())
print('JPEGImages:', (Path(IDD_ROOT)/'JPEGImages').exists(), '| Annotations:', (Path(IDD_ROOT)/'Annotations').exists())

## 2. Data prep — VOC → COCO (for RF-DETR) and → YOLO (for Ultralytics)
Images are **symlinked** (no copy), so this is fast and light on disk. All 15 IDD classes are kept.

In [ ]:
IMG_EXTS = ('.jpg', '.png', '.jpeg')

def resolve(entry):
    img = None
    for ext in IMG_EXTS:
        p = Path(IDD_ROOT)/'JPEGImages'/f'{entry}{ext}'
        if p.exists(): img = p; break
    xml = Path(IDD_ROOT)/'Annotations'/f'{entry}.xml'
    return img, (xml if xml.exists() else None)

def parse_voc(xml_path):
    r = ET.parse(xml_path).getroot()
    s = r.find('size')
    w = int(float(s.findtext('width','0'))); h = int(float(s.findtext('height','0')))
    objs = []
    for o in r.findall('object'):
        name = (o.findtext('name') or '').strip(); b = o.find('bndbox')
        if b is None or not name: continue
        x1=float(b.findtext('xmin','0')); y1=float(b.findtext('ymin','0'))
        x2=float(b.findtext('xmax','0')); y2=float(b.findtext('ymax','0'))
        if x2>x1 and y2>y1: objs.append((name,x1,y1,x2,y2))
    return w,h,objs

def read_split(sp):
    f = Path(IDD_ROOT)/f'{sp}.txt'
    if not f.exists(): return []
    out = []
    for e in f.read_text().split():
        img,xml = resolve(e)
        if img and xml: out.append((e,img,xml))
        if SUBSET and len(out) >= SUBSET: break
    return out

# stable class map across train+val
names = set()
for sp in ('train','val'):
    for _e,_i,xml in read_split(sp):
        try: names.update(n for n,*_ in parse_voc(xml)[2])
        except Exception: pass
CLASSES = sorted(names)
CAT = {n:i+1 for i,n in enumerate(CLASSES)}          # 1-indexed (COCO)
print(len(CLASSES), 'classes:', CLASSES)

In [ ]:
def link(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists(): return
    try: os.symlink(src, dst)
    except Exception: shutil.copy2(src, dst)

SPL = {'train':'train','val':'valid'}                 # IDD name -> roboflow name

def build(sp):
    rb = SPL[sp]
    cdir = COCO_DIR/rb; ydir = YOLO_DIR/rb
    cdir.mkdir(parents=True, exist_ok=True); ydir.mkdir(parents=True, exist_ok=True)
    images, anns = [], []; aid = 1
    for iid,(e,img,xml) in enumerate(read_split(sp), 1):
        w,h,objs = parse_voc(xml)
        flat = e.replace('/','__').replace('\\','__') + img.suffix
        link(img, cdir/flat); link(img, ydir/flat)
        images.append({'id':iid,'file_name':flat,'width':w,'height':h})
        ylines = []
        for name,x1,y1,x2,y2 in objs:
            cid = CAT[name]
            anns.append({'id':aid,'image_id':iid,'category_id':cid,
                         'bbox':[x1,y1,x2-x1,y2-y1],'area':(x2-x1)*(y2-y1),
                         'iscrowd':0,'segmentation':[]}); aid+=1
            if w>0 and h>0:
                ylines.append(f'{cid-1} {((x1+x2)/2)/w:.6f} {((y1+y2)/2)/h:.6f} {(x2-x1)/w:.6f} {(y2-y1)/h:.6f}')
        (ydir/(Path(flat).stem+'.txt')).write_text('\n'.join(ylines)+'\n')
    cats = [{'id':i,'name':n,'supercategory':'none'} for n,i in sorted(CAT.items(), key=lambda kv:kv[1])]
    (cdir/'_annotations.coco.json').write_text(json.dumps(
        {'images':images,'annotations':anns,'categories':cats}))
    print(f'{rb}: {len(images)} images, {len(anns)} boxes')
    return len(images)

t0=time.time()
for sp in ('train','val'): build(sp)
# YOLO data.yaml
(YOLO_DIR/'data.yaml').write_text(
    f"train: {(YOLO_DIR/'train').as_posix()}\n"
    f"val: {(YOLO_DIR/'valid').as_posix()}\n"
    f"nc: {len(CLASSES)}\nnames:\n" + ''.join(f'  {i}: {n}\n' for i,n in enumerate(CLASSES)))
print('data prep done in', round(time.time()-t0,1),'s')
print('YOLO  ->', YOLO_DIR/'data.yaml')
print('COCO  ->', COCO_DIR)

## 3. Part A — YOLOv11-l fine-tune
16 GB per T4 fits YOLOv11-l comfortably at batch 16 / imgsz 640. `device='0,1'` uses **both** T4s
(DDP); if that errors in the notebook, change to `device=0`.

In [ ]:
from ultralytics import YOLO

yolo = YOLO('yolo11l.pt')                              # COCO-pretrained, auto-downloads
yolo_res = yolo.train(
    data=str(YOLO_DIR/'data.yaml'),
    epochs=100, imgsz=640, batch=16,
    device='0,1',          # both T4s; set to 0 if DDP errors in-notebook
    workers=4, patience=25, amp=True, cache=False,
    project='/kaggle/working/yolo11l', name='idd', exist_ok=True, plots=True, verbose=True)
print('YOLOv11-l done')

In [ ]:
# YOLOv11-l metrics + per-class (auto-rickshaw is the headline India class)
m = YOLO('/kaggle/working/yolo11l/idd/weights/best.pt')
metrics = m.val(data=str(YOLO_DIR/'data.yaml'), imgsz=640, device=0)
print('mAP50    :', round(float(metrics.box.map50),4))
print('mAP50-95 :', round(float(metrics.box.map),4))
print('\nPer-class AP50:')
for i, ap in enumerate(metrics.box.ap50):
    print(f'  {CLASSES[i]:18s} {ap:.4f}')
print('\nbest.pt ->', '/kaggle/working/yolo11l/idd/weights/best.pt')

## 4. Part B — RF-DETR-Large fine-tune (the shippable detector)
RF-DETR uses its own Lightning trainer (single-GPU is most reliable in a notebook). 16 GB fits
Large at resolution 560, batch 4. Apache-2.0 — this is the deliverable model.

In [ ]:
from rfdetr import RFDETRLarge

det = RFDETRLarge()                                   # COCO-pretrained
det.train(
    dataset_dir=str(COCO_DIR), dataset_file='roboflow',
    epochs=40, batch_size=4, grad_accum_steps=4,
    resolution=560, lr=1e-4, num_workers=2,
    early_stopping=True, output_dir='/kaggle/working/rfdetr_large')
print('RF-DETR-Large done. checkpoints/metrics in /kaggle/working/rfdetr_large')

In [ ]:
# RF-DETR writes per-epoch mAP tables to the console above; final artifacts:
import glob
print('RF-DETR-Large outputs:')
for f in sorted(glob.glob('/kaggle/working/rfdetr_large/*')):
    print('  ', f)
print('\nBest weights: checkpoint_best_*.pth (Apache-2.0 → shippable spine).')

## 5. Save / download
Everything under `/kaggle/working/` is saved as the notebook **Output** — download from there:
- `yolo11l/idd/weights/best.pt`  (YOLOv11-l, AGPL — benchmark only)
- `rfdetr_large/checkpoint_best_*.pth`  (RF-DETR-Large, Apache — ship this)

**What to look for:** does fine-tuning lift mAP@0.5 from the ~0.42 zero-shot baseline toward the
DriveIndia ~0.787 reference, and does **auto-rickshaw** AP go from ~0 (zero-shot) to a real value?
That confirms the India-specific classes are now detectable.

In [ ]:
# (optional) shrink Output by removing the symlinked image dirs (weights/metrics are kept)
import shutil
for d in ('/kaggle/working/idd_yolo/train','/kaggle/working/idd_yolo/valid',
          '/kaggle/working/idd_coco/train','/kaggle/working/idd_coco/valid'):
    try: shutil.rmtree(d)
    except Exception: pass
print('cleaned symlinked image dirs (weights + metrics remain)')